# Fine-tuning open models with agents: from eval to deployment

<a target="_blank" href="https://colab.research.google.com/github/unionai/workshops/blob/main/tutorials/langgraph-grafana-agent/langgraph-grafana-agent-tutorial.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

*Durable, self-healing agents on Union, with end-to-end observability in Grafana.*

There are two agents in this notebook.

The **support agent** is the one in production. A ticket comes in, it routes it to a queue, and the queue's drafting agent writes a reply. Today the routing step is an API call per ticket: accurate, slow, and billed. We are going to replace that call with a model we own, and we are not going to pick the model ourselves.

The **ML engineer agent** is a LangGraph agent that gets this request and operates the *model factory* on Union to fill it:

> Replace the support agent's router with a self-hosted open-source model: at least 95% accuracy on our tickets, under 150 ms per ticket on a T4. Baseline up to 5 candidates, fine-tune up to 3 of them, spend at most 12 runs.

| Step | You will | Minutes |
|---|---|---|
| 0 | Run the support agent as it is today. The "before". | 3 |
| 1 | Drive the factory yourself, no agent: parallel GPU evals, one fine-tune. | 5 |
| 2 | Hand the request to the ML engineer agent. It builds, deploys and tests the router. | 5 |
| 3 | Switch the support agent to the new router. The "after". | 3 |
| 4 | Look at both agents in Grafana. | 5 |
| 5 | Drift: a new kind of ticket arrives. See it. | 3 |
| 6 | The loop closes itself: publish the tickets, a trigger notices, the engineer retrains. | 10 |
| 7 | Bake off the engineer's brain, including an open model (optional). | 5 |
| 8 | Kill the engineer mid-run and watch it resume (optional). | 5 |

Every step is a Flyte task. `run(task)` sends it to the cluster, prints the run URL, waits, and returns the result. **Open every run URL**: the run graph, the reports, and the Grafana links are the point. The `view_file(...)` cells show the code behind each step; read them when you want to know how, skip them when you want to keep moving.

---

## Setup

Run the next cell once. On Colab it clones the repo, installs the dependencies (a minute or two), and sets up a plain-text keyring so the cluster login can store its token. Locally, with the repo cloned and a virtualenv from `requirements.txt`, it only does the imports.

Two helpers come out of it: `run`, which submits a task to the cluster and waits, and `view_file`, which renders a source file inline.

In [ ]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !git clone -q https://github.com/unionai/workshops
    %cd workshops/tutorials/langgraph-grafana-agent
    !pip install -q uv
    !uv pip install --system -q -r requirements.txt
    !uv pip install --system -q keyrings.alt pygments
    !mkdir -p ~/.config/python_keyring && echo -e '[backend]\ndefault-keyring=keyrings.alt.file.PlaintextKeyring' > ~/.config/python_keyring/keyringrc.cfg
    %env TERM=dumb

import os
from getpass import getpass
from utils.file_viewer import view_file
from utils.workshop import run, show

### Connect to the cluster

You were given a Union endpoint and a project. This writes the config the SDK and the `flyte` CLI read. The first cluster call after it prints a login URL and a code: open the URL, sign in, paste the code back into the cell. Colab has no browser to hand off to, which is why the login is `headless`.

Everyone in the room shares this project. Fine-tunes are cached per project, so after the first person trains a model nobody trains it again; evals run every time, so every run shows its work. The serving app and the artifacts are shared too, which means the last promotion wins. If you would rather have your own, set `os.environ["FACTORY_TAG"] = "yourname"` before the next cell and everything you deploy carries that suffix.

In [ ]:
ENDPOINT = "tryv2.hosted.unionai.cloud"
PROJECT = "workshopgrafana"

!flyte create config --endpoint {ENDPOINT} --project {PROJECT} --domain development --builder remote --auth-type headless --force

### The agent's model (optional)

Both agents think with Claude by default (`anthropic:claude-opus-5`). The key lives on the cluster as a secret, so nothing is needed here for cluster runs. Uncomment to drive them with OpenAI instead, or to run a step locally with `run(task, local=True)`, which needs the key in this process.

In [ ]:
# To use OpenAI instead of Claude:
# os.environ["AGENT_MODEL"] = "openai:gpt-4.1"
# os.environ["FACTORY_PROVIDERS"] = "openai"

# Only for local runs (run(task, local=True)); cluster runs read the secret instead:
# os.environ["ANTHROPIC_API_KEY"] = getpass("ANTHROPIC_API_KEY: ")

### The API key on the cluster

Tasks read the key from a Union secret named after the provider: `ANTHROPIC_API_KEY`, or `OPENAI_API_KEY`. The workshop project already has one. If you are on your own cluster, create it once (the `--value` flag matters: without it the command waits for input that never comes in a notebook).

In [ ]:
# !flyte create secret ANTHROPIC_API_KEY --value sk-ant-... --project {PROJECT} --domain development
!flyte get secret --project {PROJECT} --domain development

### Grafana (optional)

If you have a Grafana Cloud stack with Agent Observability enabled, put its values in the environment and every run from here on exports to it: each agent run becomes a conversation, each tool call a step, and the router app reports its own metrics. If not, skip this: everything else works, and the host will show theirs.

In [ ]:
# os.environ["GRAFANA_HOST"] = "https://<stack>.grafana.net"
# os.environ["AGENTO11Y_ENDPOINT"] = "https://agento11y-prod-<region>.grafana.net"     # Agent Observability -> Configuration
# os.environ["AGENTO11Y_AUTH_TENANT_ID"] = "<instance id>"
# os.environ["OTEL_EXPORTER_OTLP_ENDPOINT"] = "https://otlp-gateway-prod-<region>.grafana.net/otlp"
# os.environ["GRAFANA_TOKEN"] = getpass("GRAFANA_TOKEN (glc_...): ")
# !flyte create secret GRAFANA_TOKEN --value {os.environ["GRAFANA_TOKEN"]} --project {PROJECT} --domain development

---

## 0. The support agent, as it is today

Thirty held-out tickets go through the support agent: the router picks a queue, then the queue's drafting agent (the same model, prompted as that team) writes a two-line reply. Open the run and read the report: routing accuracy, p50 route latency, tokens, and cost per 1,000 tickets. This is the "before".

What we measured on the same thirty tickets, so you know what to expect (cost is routing plus replies, at list price):

| Support agent model | Routing accuracy | Route p50 | Per 1,000 tickets |
|---|---|---|---|
| Claude Opus 5 (the default) | 96.7% | 2.28 s | $10.28 |
| Claude Haiku 4.5 | 96.7% | 596 ms | $1.18 |
| GPT-4.1 | 93.3% | 594 ms | $0.64 |

Opus thinks before it routes, so it is slower at the same accuracy. To try a cheaper model, pass `model="anthropic:claude-haiku-4-5"` to `run`.

In [ ]:
from support_agent import agent_handle_tickets

before = run(agent_handle_tickets, n=30, router="llm")
show(before, ["routing_accuracy", "route_p50_ms", "cost_per_1000_tickets_usd", "tokens"])

**What to look at.** In the run graph, every ticket is two traced steps, `route_with_model` and `draft_reply`. Click one: the inputs are the ticket text and the model name, the output is the queue or the reply. Because each step is recorded, a batch that dies halfway replays what it already did instead of paying for it twice.

**How it is built.** Two LangGraph nodes, `route → draft`, in a single Flyte task. `router="llm"` is today's routing, an API call per ticket. `router="oss"` calls the app we are about to build.

In [ ]:
view_file("support_agent.py", show_path=True)

---

## 1. The factory, with you at the controls

Before the engineer touches anything, drive the factory yourself. This one task baselines three candidate models on 120 held-out tickets, as three T4 containers running at the same time, then fine-tunes one of them for an epoch and evaluates it again.

Open the run while it works. The three evals sit side by side under `baseline-evals`, each named after its model (`run_eval · qwen2.5-0.5b`), each with its own report. The fine-tune under `fine-tune-and-recheck` draws its loss curve live. The parent report is the eval table.

The fine-tune is cached by its inputs: when the engineer asks for the same one in step 2, it gets the weights back in seconds, and so does everyone else in the room. The evals are not cached, on purpose, so the engineer measures again and you watch it happen.

In [ ]:
from model_factory import model_factory

run(model_factory, fine_tune_too=True)

### The candidates and the numbers

Five candidates, all Apache-2.0, from 149M to 1.7B parameters. Measured on the same 120 tickets, on a T4:

| candidate | zero-shot | after fine-tune | p50 on T4 |
|---|---|---|---|
| smollm2-360m | 13% | | 110 ms |
| qwen2.5-0.5b | 59% | 1 epoch: 97.5% | 68 ms |
| qwen2.5-1.5b | 79% | 1 epoch: 97.5% | 83 ms |
| smollm2-1.7b | 36% | | 72 ms |
| modernbert-base (an encoder, not a chat model) | 9%, untrained head | 3 epochs, 16 s: 99.2% | 14 ms |

Nothing passes zero-shot. The bar is 95%. The engineer has to work that out from numbers it produces itself, and it has to notice that the one model that is not a chat model at all turns out to be the best router once it is trained.

For scale, the models the support agent could route with today, zero-shot on the same 120 tickets (p50 is a network call from the cluster):

| router | zero-shot | p50 | where it runs |
|---|---|---|---|
| Claude Opus 5 | 98.3% | 1.7 s | API |
| GPT-4.1 | 97.5% | 634 ms | API |
| Claude Haiku 4.5 | 95.8% | 606 ms | API |
| Qwen3-8B (vLLM) | 94.2% | 386 ms | one L40s, ours |
| modernbert-base, fine-tuned | 99.2% | 14 ms on a T4 | 149M params, ours |

The big API models clear the bar without training; the trained encoder beats all of them at a hundredth of the latency.

**How it is built.** `factory.py` is plain transformers, peft and trl: `evaluate()` and `fine_tune()`, no Flyte in it. `tools.py` wraps those functions as Flyte tasks with the GPU, the cache, the retries and the reports, and those tasks are the tools the engineer gets.

In [ ]:
view_file("factory.py", show_path=True)

In [ ]:
view_file("tools.py", show_path=True)

---

## 2. Hand the request to the ML engineer agent

Now the same factory, operated by an agent. The engineer's graph is `think → tools → think … → decision`: the model reads the request, calls tools, reads the results, and keeps going until it has something to promote or has spent the budget. Every model turn is a recorded step; every tool call is a durable child action on its own container. Independent calls it asks for in one turn run in parallel, so "evaluate these four" is four T4s at once.

Open the run while it works. Every action is named after what it does (`run_eval · qwen2.5-0.5b`, `fine_tune · modernbert-base · 3ep`). Each `think:model` step shows the turn number and the message the model is reacting to as inputs, and what it said and asked for as outputs. At the end, `promote` publishes the winner as the `ticket-router` artifact and deploys the serving app, sized to the model it chose, and `test_deployment` calls the live app with tickets it has not seen. The parent report is the decision page: what was promoted, pass or fail against the request, everything it measured, the sequence of calls, and the rationale in the agent's own words.

This takes four to five minutes; most of it is the evals and fine-tunes on T4s, which you can watch in the run graph.

In [ ]:
from ml_engineer import ml_engineer_agent

decision = run(ml_engineer_agent)
show(decision["decision"], ["action", "promoted_model", "accuracy", "latency_p50_ms", "deployment_test_passed", "runs_spent"])

**What just happened.** The agent had eight tools and a budget of twelve runs. It baselined the candidates, found nothing that passed, fine-tuned the ones worth it (in parallel), picked the smallest model that cleared both bars, promoted it, and tested the deployment. Nothing in that sequence was scripted; the order and the choices are in the report.

**How it is built.** `graph.py` holds the request, the typed `Decision` the run ends with, the parallel tool node, and the `think` step. The tools are the tasks from `tools.py`. The scoring in the report checks the agent's claims against its own measurements, because an agent that says "94.2% rounds to 95" is a thing that happens.

In [ ]:
view_file("graph.py", show_path=True)

---

## 3. Switch the support agent to the new router

Same thirty tickets, same draft replies, but routing is now an HTTP call to the app the engineer deployed. Nothing in the support agent changed except a flag. Put this report next to step 0's: accuracy holds or improves, latency drops by an order of magnitude, and the cost that remains is the reply drafts.

What we measured, before → after:

| Support agent model | Routing accuracy | Route p50 | Per 1,000 tickets |
|---|---|---|---|
| Claude Opus 5 | 96.7% → 100% | 2.28 s → 128 ms | $10.28 → $6.47 |
| Claude Haiku 4.5 | 96.7% → 100% | 596 ms → 170 ms | $1.18 → $0.30 |
| GPT-4.1 | 93.3% → 96.7% | 594 ms → 200 ms | $0.64 → $0.34 |

In [ ]:
after = run(agent_handle_tickets, n=30, router="oss")
show({"before": {k: before[k] for k in ("routing_accuracy", "route_p50_ms", "cost_per_1000_tickets_usd")},
      "after":  {k: after[k]  for k in ("routing_accuracy", "route_p50_ms", "cost_per_1000_tickets_usd")}})

**How it is built.** The router app is a FastAPI service on Union that mounts the promoted artifact at startup and classifies on `/classify`. The pod is sized to the model: a small CPU pod for the encoder, a T4 for a larger chat model, so the latency the request was judged on is the latency production sees. With Grafana configured it also reports predictions per label, confidence and latency, which is the drift dashboard in step 5.

In [ ]:
view_file("router_app.py", show_path=True)

---

## 4. Both agents in Grafana

Nothing to run. Open the step 2 run in the Flyte UI: the task carries two links. **Grafana Agent Observability** opens the run as a conversation: every model turn with its prompt, answer, model, tokens and cost, every tool call as a step, the totals in the header. **Grafana trace** opens the same run in Tempo, where a `fine_tune` span is a minute and a half wide and the three of them overlap. The support agent's runs from steps 0 and 3 are conversations too, side by side: one full of routing generations, one without them.

**How it is built.** The whole integration is one `init()` call at module scope in `config.py`, plus the instrumentation the LangGraph plugin adds to every model call.

In [ ]:
view_file("config.py", show_path=True)

---

## 5. Drift: a new kind of ticket

The support team adds a ninth queue, `data_request`, for GDPR-style tickets ("delete my account and all my data", "send me everything you have on me"). Your router has never seen the label. See it first: the support agent on the v2 tickets, still routing with the model deployed in step 2.

The GDPR tickets pile up in `other` at low confidence, and the report flags them as tickets that belong to a queue the router does not know. If you have Grafana, the router's own metrics show the same thing: the share of `other` climbs and confidence drops. This is what drift looks like from the outside, and it is the reason a deployed model needs a loop around it.

In [ ]:
drift = run(agent_handle_tickets, n=36, router="oss", dataset="v2")
show(drift, ["routing_accuracy", "tickets_with_unknown_queue", "queue_distribution"])

---

## 6. The loop closes itself

Nobody should have to notice that. Deploy two triggers once. The first watches the `support-tickets` artifact: whenever a new version lands, a run of `adapt` starts that you did not start. It is the same engineer, told "new tickets landed, check production first". It measures the live model on the new tickets, finds it below the bar, retrains on them, promotes, and tests the deployment. That promotion fires the second trigger, which validates every new `ticket-router` version on a T4, whoever published it. If production had still met the bar, `adapt` would have kept it and stopped.

In [ ]:
!flyte deploy adaptive_loop.py observed_env
!flyte deploy validate_on_promote.py validate_env

Now publish the v2 tickets. Then open the runs list in the Flyte UI and watch: a run of `adapt` appears on its own (a minute or so), then a run of `validate_router`. Each firing also leaves a run named `at-<trigger>` with no inputs or report: that is the trigger's own record, and the work is in the task run next to it. Open the `adapt` run: it is a full decision page, made without you.

In [ ]:
from adaptive_loop import publish_tickets

run(publish_tickets, version="v2")
print("watch the runs list: 'adapt' appears on its own, then 'validate_router'")

When the `adapt` run has finished, the support agent is fine again. The `ticket-router` artifact has a new version; open it in the UI and look at Versions and Lineage: from the app, back through the promotion, to the fine-tune, to the tickets that caused it.

In [ ]:
after_fix = run(agent_handle_tickets, n=36, router="oss", dataset="v2", labels_from="v2")
show(after_fix, ["routing_accuracy", "tickets_with_unknown_queue", "queue_distribution"])

### The same fix, by hand (optional)

If you would rather watch the engineer do it in front of you, this is the same request without the trigger: tell the engineer what changed and let it work.

In [ ]:
# from drift import day_two
# fixed = run(day_two)
# show(fixed["decision"], ["action", "promoted_model", "accuracy", "deployment_test_passed", "runs_spent"])

---

## 7. Bake off the engineer's brain (optional)

The engineer's own model is a model too, and it can be graded like any other. Same request, several models driving the engineer in parallel, scored on whether they promoted something that meets it and how many runs they spent. Each agent model is its own agent version in Grafana, so the conversations sit side by side. Promotion publishes artifacts here but does not redeploy the router. Skip if short on time.

One contender can be an open model: Qwen3-8B served by vLLM on a Union app with an L40s (`python serve_model.py` deploys it once for a whole room). On our cluster it clears the bar too, promoting the fine-tuned 1.5B at 100% and 84 ms while using all 12 runs, so the ML engineer can be off the API as well, and the whole loop, brain included, runs on models you own.

In [ ]:
# from bakeoff import bakeoff
# run(bakeoff, models=[os.environ.get("AGENT_MODEL", "anthropic:claude-opus-5"), "anthropic:claude-haiku-4-5"])

# With the host's vLLM app (an OpenAI-compatible endpoint on Union), add the open model:
# os.environ["VLLM_BASE_URL"] = "https://<the host's vllm app>/v1"
# os.environ["VLLM_API_KEY_SECRET_NAME"] = "VLLM_API_KEY"        # the secret holding the app's key
# os.environ["FACTORY_PROVIDERS"] = "anthropic,vllm"
# run(bakeoff, models=["anthropic:claude-opus-5", "vllm:qwen3-8b"])

---

## 8. Kill the engineer, watch it resume (optional)

A durable run is not one process. This is the same engineer with a booby-trapped model: it dies after its third live model call on the first attempt, right after the fine-tunes come back. Flyte retries it in a fresh container, where the model turns it already paid for replay from their records, the evals and fine-tunes are cache hits, and the decision is produced once.

In the run graph, expect exactly one red `think:model`: that is the crash itself, with the message `simulated worker crash after 3 model calls`. Everything after it is attempt 1. In the pod logs, attempt 0 prints three `live model call` lines and the crash; attempt 1 prints four. In Grafana both attempts are one trace, with the replayed steps marked.

In [ ]:
# from crash_resume import resilient_engineer
# run(resilient_engineer)

---

## Where to go from here

- Change the request: `run(ml_engineer_agent, min_accuracy=0.98, max_latency_ms=50)` is a different problem.
- Add a candidate to `factory.CANDIDATES`. Anything on the Hub with a chat template, or any encoder, works.
- Replace `tickets.py` with your own data. The tools do not care where the tickets come from.
- Turn on the approval gate: `os.environ["FACTORY_APPROVAL"] = "1"` and `promote` pauses in the Flyte UI until someone says yes.
- The ticket router is one example. The same loop works for a reranker, a fraud scorer, a PII detector, or an intent model: anywhere an agent leans on a frontier model for a narrow job and you have the data.
- The README has the full write-up, the timings, and the gotchas we hit building this.